In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
# import os
# from pathlib import Path

notebook_path = "/u/skarmakar1/version_check/llm_steering-main/sk"
sys.path.append(notebook_path)

from inversion_utils import *

In [3]:
import torch
import numpy as np
# from transformers import AutoTokenizer, AutoModelForCausalLM, MllamaForConditionalGeneration, AutoProcessor
# from transformers import BitsAndBytesConfig
# from transformers import Gemma3ForCausalLM

# # from neural_controllers import NeuralController
# from neural_controllers_xrfm import NeuralController

# from utils import LLMType
# from collections import namedtuple 
# from tqdm import tqdm 
# import gc
# import os

In [4]:
# from janus.utils.io import load_pil_images
# from generation_utils import extract_image
import utils

In [5]:
SEED = 0
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
np.random.seed(SEED)

In [6]:
# LLM = namedtuple('LLM', ['language_model', 'tokenizer', 'processor', 'name', 'model_type'])


# def select_llm(model_type, MODEL_VERSION='3.1', MODEL_SIZE='8B'):

#     custom_cache_dir = "/scratch/bbjr/skarmakar/huggingface"

#     if model_type=='llama':

#         if MODEL_VERSION == '3.1' and MODEL_SIZE == '8B':
#             model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"
#         elif MODEL_VERSION == '3.1' and MODEL_SIZE == '70B':
#             model_id = "unsloth/Meta-Llama-3.1-70B-Instruct-bnb-4bit"
#         elif MODEL_VERSION == '3.3' and MODEL_SIZE == '70B':
#             model_id = "unsloth/Llama-3.3-70B-Instruct-bnb-4bit"

#         language_model = AutoModelForCausalLM.from_pretrained(
#             model_id, device_map="cuda", cache_dir=custom_cache_dir,
#         )

#         use_fast_tokenizer = "LlamaForCausalLM" not in language_model.config.architectures
#         tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=use_fast_tokenizer, padding_side="left", legacy=False)
#         tokenizer.pad_token_id = 0 
#         # model_name='llama_3_8b_it'
#         if MODEL_VERSION == '3.1' and MODEL_SIZE == '8B':
#             model_name='llama_3_8b_it_eng_only'
#         elif MODEL_VERSION == '3.1' and MODEL_SIZE == '70B':
#             model_name = "llama_3.1_70b_it_eng_only"
#         elif MODEL_VERSION == '3.3' and MODEL_SIZE == '70B':
#             model_name = "llama_3.3_70b_it_eng_only"

#         processor = None
#         llm_type = LLMType.TEXT

#         language_model.generation_config.pad_token_id = tokenizer.pad_token_id # to disable the warning

#         language_model.generation_config.temperature=None # to disable the stupid warnings
#         language_model.generation_config.top_p=None # to disable the stupid warnings
#         # language_model.generation_config.top_k=None # to disable the stupid warnings

#     elif model_type=='gemma':
#         quantization_config = BitsAndBytesConfig(load_in_8bit=True)
#         if MODEL_VERSION == '3' and MODEL_SIZE == '1B':
#             model_id = "google/gemma-3-1b-it"
#         elif MODEL_VERSION == '3' and MODEL_SIZE == '12B':
#             model_id = "google/gemma-3-12b-it"
        

#         tokenizer = AutoTokenizer.from_pretrained(model_id)

#         # language_model = Gemma3ForCausalLM.from_pretrained(
#         #     model_id, quantization_config=quantization_config
#         # ).eval()

#         # language_model = Gemma3ForCausalLM.from_pretrained(
#         #     model_id, torch_dtype=torch.float16, cache_dir=custom_cache_dir,
#         # ).eval()

#         language_model = Gemma3ForCausalLM.from_pretrained(
#             model_id, quantization_config=quantization_config, cache_dir=custom_cache_dir,
#         ).eval()
        

#         if MODEL_VERSION == '3' and MODEL_SIZE == '1B':
#             model_name='gemma_3_1b_it_eng_only'
#         elif MODEL_VERSION == '3' and MODEL_SIZE == '12B':
#             model_name='gemma_3_12b_it_eng_only'

#         processor = None
#         llm_type = LLMType.GEMMA_TEXT

#         # print(tokenizer.chat_template)

#     llm = LLM(language_model, tokenizer, processor, model_name, llm_type)
#     # print(llm.language_model)
#     return llm


# def compute_save_directions_translate(llm, dataset, concept, control_method='rfm', orig_lang="", method='lstsq', path='directions/'):

#     if os.path.exists(os.path.join(path, f'{control_method}_{orig_lang}TO{concept}_{llm.name}.pkl')):
#         print(f"'{os.path.join(path, f'{control_method}_{orig_lang}TO{concept}_{llm.name}.pkl')}' exists, skipping it.")
#         return

    
#     concept_types = [concept]
#     for concept_type in concept_types:
#         controller = NeuralController(
#             llm,
#             llm.tokenizer,
#             rfm_iters=8,
#             control_method=control_method,
#             n_components=1,
#         )
#         controller.compute_directions(dataset[concept_type]['train']['inputs'], dataset[concept_type]['train']['labels'], method='lstsq')

#         controller.save_translate(concept=f'{concept_type}', model_name=llm.name, orig_lang=orig_lang, path=path)        



# def read_file(fname, lower=True):

#     concepts = []
#     with open(fname, encoding="utf-8") as f: 
#         for line in f:
#             if lower:
#                 concepts.append(line.strip().lower())
#             else:
#                 concepts.append(line.strip())
#     concepts = sorted(list(set(concepts)))
#     return concepts

In [7]:
torch.backends.cudnn.benchmark = True 
torch.backends.cuda.matmul.allow_tf32 = True

In [8]:
# model_type = 'llama'
# MODEL_VERSION='3.1'
# MODEL_SIZE='8B'

# model_type = 'gemma'
# # MODEL_VERSION='2'
# MODEL_VERSION='3'
# # MODEL_SIZE='1B'
# # MODEL_SIZE='9B'
# MODEL_SIZE='12B'

model_type = 'qwen'
MODEL_VERSION='3'
# MODEL_SIZE='4B'
# MODEL_SIZE='8B'
MODEL_SIZE='30B'



llm = select_llm(model_type, MODEL_VERSION=MODEL_VERSION, MODEL_SIZE=MODEL_SIZE)

Loading checkpoint shards:   0%|          | 0/16 [00:00<?, ?it/s]

In [9]:
hidden_layers = list(range(-1, -llm.language_model.config.num_hidden_layers, -1))
print(hidden_layers)

[-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31, -32, -33, -34, -35, -36, -37, -38, -39, -40, -41, -42, -43, -44, -45, -46, -47]


Single

In [10]:
source_lang = 'english'
target_lang = 'korean'

save_path = 'all_gitignore/xRFM/test/'

METHOD = 'rfm'

solver = 'lstsq'

print(f"============================================={source_lang} to {target_lang}=============================================")

dataset = utils.multilingual_dataset(llm, source_lang, target_lang, n=600)

compute_save_directions_translate(llm, dataset, target_lang, control_method=METHOD, orig_lang=source_lang, solver=solver, path=save_path)

del dataset
torch.cuda.empty_cache()

gc.collect()

=============================================english to korean=============================================
Loading FLORES-200 for english (eng_Latn) and korean (kor_Hang)...
train 1200
Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31, -32, -33, -34, -35, -36, -37, -38, -39, -40, -41, -42, -43, -44, -45, -46, -47]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components         : 1

Using xRFM library for RFM-based steering vector extraction
use_concat False


 95%|█████████▍| 71/75 [00:57<00:03,  1.19it/s]

100%|██████████| 75/75 [01:00<00:00,  1.24it/s]



Using lstsq sovler for xRFM!!



  2%|▏         | 1/47 [00:02<02:03,  2.68s/it]

...

Best xRFM r: 0.9890273213386536, reg: 0.001, bw: 10, norm: False


100%|██████████| 47/47 [00:45<00:00,  1.04it/s]


Best xRFM r: 0.9872129559516907, reg: 0.001, bw: 100, norm: False


100%|██████████| 47/47 [00:00<00:00, 96586.13it/s]


12

In [14]:
coef = 0.45
# coef = 0.65
# coef = 0.7 # llama
# coef = 0.75

# coef = 2.0
# coef = 2.5 # qwen
# coef = 3.0
# coef = 3.5

# coef = 65.0
# coef = 80.0


# path = "../all_gitignore/xRFM/language_multiex_llama/"
# path = "../all_gitignore/xRFM/language_multi_gemma"
# path = "../all_gitignore/xRFM/language_multi_qwen"
# path = "../all_gitignore/xRFM/language_multiex_qwen"
path = "all_gitignore/xRFM/test"

max_tokens = 200
# max_tokens = 100

# prompts = ["How does a clock work?"] # "english"
prompts = ["What is weight of a football used in fifa?"] # "english"
# prompts = ["Quel est le poids d'un ballon de football utilisé par la FIFA ?"] # "french"
# prompts = ["फीफा में इस्तेमाल होने वाली फुटबॉल का वज़न कितना होता है?"] # "hindi"
# prompts = ["¿Cuál es el peso de un balón de fútbol utilizado en la FIFA?"] # 'spanish'


orig_lang = "english"
# orig_lang = "hindi"
# orig_lang = "german"
# orig_lang = 'spanish'
# orig_lang = 'french'

# l = "english"
# l = "german"
# l = "hindi"
# l = "thai"
# l = 'italian'
# l = 'french'
# l = 'portuguese'

# l = "czech"
l = "korean"

l_controller = load_controller_translate(llm, l, orig_lang, path=path)

# out = test_concept_vector(l_controller, concept=f"coef: {coef}, lang: {l}", prompts=prompts, coef=coef, max_tokens=max_tokens, qwen=False, orig=False)
out = test_concept_vector(l_controller, concept=f"coef: {coef}, lang: {l}", prompts=prompts, coef=coef, max_tokens=max_tokens, qwen=True, orig=False)


# for l in all_langs:
#     if l == orig_lang or l == "italian":
#         continue

#     # l_controller = load_controller(llm, l, path=f'../all_gitignore/language_multi/')
#     l_controller = load_controller_translate(llm, l, orig_lang, path=path)

#     # out = test_concept_vector(l_controller, concept=f"coef: {coef}, lang: {l}", prompts=prompts, coef=coef, max_tokens=max_tokens, qwen=False, orig=False)
#     out = test_concept_vector(l_controller, concept=f"coef: {coef}, lang: {l}", prompts=prompts, coef=coef, max_tokens=max_tokens, qwen=True, orig=False)

Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31, -32, -33, -34, -35, -36, -37, -38, -39, -40, -41, -42, -43, -44, -45, -46, -47]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components         : 1

Using xRFM library for RFM-based steering vector extraction
Detector found

========================== + coef: 0.45, lang: korean Control (normal) ==========================
  
The weight of a football used in FIFA (International Football Federation) matches is **410~450그램**입니다.  

정확히는 FIFA 공식 규정에 따라 **공식 규격**은 다음과 같습니다:  
- **공식 명칭**: Size 5 (공식적으로는 "Size 5"로 표기됨)  
- **무게**: **410g ~ 450g**  
- **지름**: 약 22cm (21.8~22.2cm)  
- **내부 압력**: 0.6~1.1 기압 (600~1100 토르)  

이 규격은 FIFA 공식 규정에 따라 **공식적으로 승인된 공**만이 사용 가능하며, 한국에서 판매되는 대부분의 축구공도 이 범위 내에 있습니다.  

>


In [ ]:
# all_langs = ['english', 'german', 'hindi', 'spanish', 'french', 'italian', 'portuguese', 'thai', 'chinese_simplified', 'chinese_traditional', 'japanese', 'korean', 'russian', 'arabic', 'vietnamese', 'indonesian', 'turkish', 'polish', 'dutch', 'ukrainian', 'czech', 'romanian', 'greek', 'hungarian', 'swedish', 'danish', 'finnish', 'norwegian', 'bulgarian', 'serbian', 'croatian', 'slovak', 'lithuanian', 'slovenian', 'latvian', 'estonian', 'catalan', 'hebrew', 'persian', 'tagalog', 'bengali', 'urdu', 'tamil', 'telugu', 'malayalam', 'kannada', 'marathi', 'gujarati', 'punjabi', 'malay', 'swahili', ]

# # all_langs.reverse()

In [10]:
all_langs = ['english', 'german', 'hindi', 'spanish', 'french', 'italian', 'portuguese', 'thai', 'chinese_simplified', 'chinese_traditional', 'japanese', 'korean', 'russian', 'arabic', 'vietnamese', 'indonesian', 'turkish', 'polish', 'dutch', 'ukrainian', 'czech',]

In [ ]:
source_langs = ['english',]
target_langs = ['italian', ]

# test_set = ['portuguese', 'japanese',]

# other_langs = [element for element in all_langs if element not in source_langs + target_langs + test_set]
other_langs = [element for element in all_langs if element not in source_langs + target_langs]

# original_langs = ['english', 'french', 'german', 'hindi', 'italian', 'portuguese', 'spanish', 'thai']
# other_langs = ['english', 'french', 'german', 'hindi', 'italian', 'portuguese', 'spanish', 'thai']
# # other_langs = ['english', 'french', 'hindi']



# # dataset_label = 'language'
# dataset_label = 'language_multi'

# save_path = 'all_gitignore/language/'
# save_path = 'all_gitignore/language_multi/'


save_path = 'all_gitignore/xRFM/language_multiex_llama/'
# save_path = 'all_gitignore/xRFM/language_multi_gemma/'
# save_path = 'all_gitignore/xRFM/language_multi_qwen/'
# save_path = 'all_gitignore/xRFM/language_multiex_qwen/'

# save_path = 'all_gitignore/xRFM/test/'
# save_path = 'all_gitignore/xRFM/test_eigenpro/'

# save_path = 'all_gitignore/xRFM/language_multi/'


# METHOD = 'mean_difference'
METHOD = 'rfm'

solver = 'lstsq'
# solver = 'eigenpro'


In [13]:
for source_lang in source_langs:
    for other_lang in other_langs:
        print(f"============================================={source_lang} to {other_lang}=============================================")
        
        # if dataset_label == 'language':
        #     dataset = utils.language_dataset(llm, original_lang, other_lang)
        # elif dataset_label == 'language_multi':
        #     dataset = utils.multilingual_dataset(llm, original_lang, other_lang)

        dataset = utils.multilingual_dataset(llm, source_lang, other_lang)
        
        compute_save_directions_translate(llm, dataset, other_lang, control_method=METHOD, orig_lang=source_lang, solver=solver, path=save_path)
        
        del dataset
        torch.cuda.empty_cache()

        gc.collect()

    #     break
    # break

for target_lang in target_langs:
    for other_lang in other_langs:
        print(f"============================================={target_lang} to {other_lang}=============================================")

        dataset = utils.multilingual_dataset(llm, target_lang, other_lang)
        
        compute_save_directions_translate(llm, dataset, other_lang, control_method=METHOD, orig_lang=target_lang, solver=solver, path=save_path)
        
        del dataset
        torch.cuda.empty_cache()

        gc.collect()

    #     break
    # break

=============================================english to german=============================================
Loading FLORES-200 for english (eng_Latn) and german (deu_Latn)...
train 400
'all_gitignore/xRFM/language_multiex_llama/rfm_englishTOgerman_llama_3_8b_it_eng_only.pkl' exists, skipping it.
=============================================english to hindi=============================================
Loading FLORES-200 for english (eng_Latn) and hindi (hin_Deva)...
train 400
'all_gitignore/xRFM/language_multiex_llama/rfm_englishTOhindi_llama_3_8b_it_eng_only.pkl' exists, skipping it.
=============================================english to spanish=============================================
Loading FLORES-200 for english (eng_Latn) and spanish (spa_Latn)...
train 400
'all_gitignore/xRFM/language_multiex_llama/rfm_englishTOspanish_llama_3_8b_it_eng_only.pkl' exists, skipping it.
=============================================english to french=============================================
L

In [14]:
for source_lang in other_langs:
    for other_lang in source_langs:
        print(f"============================================={source_lang} to {other_lang}=============================================")
        
        # if dataset_label == 'language':
        #     dataset = utils.language_dataset(llm, original_lang, other_lang)
        # elif dataset_label == 'language_multi':
        #     dataset = utils.multilingual_dataset(llm, original_lang, other_lang)

        dataset = utils.multilingual_dataset(llm, source_lang, other_lang)
        
        compute_save_directions_translate(llm, dataset, other_lang, control_method=METHOD, orig_lang=source_lang, solver=solver, path=save_path)
        
        del dataset
        torch.cuda.empty_cache()

        gc.collect()

    #     break
    # break

for target_lang in other_langs:
    for other_lang in target_langs:
        print(f"============================================={target_lang} to {other_lang}=============================================")

        dataset = utils.multilingual_dataset(llm, target_lang, other_lang)
        
        compute_save_directions_translate(llm, dataset, other_lang, control_method=METHOD, orig_lang=target_lang, solver=solver, path=save_path)
        
        del dataset
        torch.cuda.empty_cache()

        gc.collect()

    #     break
    # break

=============================================german to english=============================================
Loading FLORES-200 for german (deu_Latn) and english (eng_Latn)...
train 400
Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components         : 1

Using xRFM library for RFM-based steering vector extraction
use_concat False


100%|██████████| 25/25 [00:04<00:00,  5.81it/s]



Using lstsq sovler for xRFM!!



  3%|▎         | 1/31 [00:01<00:51,  1.71s/it]

Best xRFM r: 0.994522213935852, reg: 0.001, bw: 100, norm: False


...

Best xRFM r: 0.9786472320556641, reg: 0.001, bw: 10, norm: True


100%|██████████| 31/31 [01:17<00:00,  2.51s/it]


Best xRFM r: 0.9667821526527405, reg: 0.001, bw: 100, norm: True


100%|██████████| 31/31 [00:00<00:00, 83294.95it/s]


=============================================german to italian=============================================
Loading FLORES-200 for german (deu_Latn) and italian (ita_Latn)...
train 400
'all_gitignore/xRFM/language_multiex_llama/rfm_germanTOitalian_llama_3_8b_it_eng_only.pkl' exists, skipping it.
=============================================hindi to italian=============================================
Loading FLORES-200 for hindi (hin_Deva) and italian (ita_Latn)...
train 400
'all_gitignore/xRFM/language_multiex_llama/rfm_hindiTOitalian_llama_3_8b_it_eng_only.pkl' exists, skipping it.
=============================================spanish to italian=============================================
Loading FLORES-200 for spanish (spa_Latn) and italian (ita_Latn)...
train 400
'all_gitignore/xRFM/language_multiex_llama/rfm_spanishTOitalian_llama_3_8b_it_eng_only.pkl' exists, skipping it.
=============================================french to italian=============================================
L

Tests

In [17]:
# coef = 0.65
# coef = 0.7 # llama
# coef = 0.75

coef = 2.0
# coef = 2.5 # qwen
# coef = 3.0
# coef = 3.5

# coef = 65.0
# coef = 80.0


path = "all_gitignore/xRFM/test"

max_tokens = 200
# max_tokens = 100

# prompts = ["How does a clock work?"] # "english"
prompts = ["What is weight of a football used in fifa?"] # "english"
# prompts = ["Quel est le poids d'un ballon de football utilisé par la FIFA ?"] # "french"
# prompts = ["फीफा में इस्तेमाल होने वाली फुटबॉल का वज़न कितना होता है?"] # "hindi"
# prompts = ["¿Cuál es el peso de un balón de fútbol utilizado en la FIFA?"] # 'spanish'


orig_lang = "english"
# orig_lang = "hindi"
# orig_lang = "german"
# orig_lang = 'spanish'
# orig_lang = 'french'

# l = "english"
# l = "german"
# l = "hindi"
# l = "thai"
# l = 'italian'
# l = 'french'
# l = 'portuguese'

l = "korean"

l_controller = load_controller_translate(llm, l, orig_lang, path=path)

# out = test_concept_vector(l_controller, concept=f"coef: {coef}, lang: {l}", prompts=prompts, coef=coef, max_tokens=max_tokens, qwen=False)
out = test_concept_vector(l_controller, concept=f"coef: {coef}, lang: {l}", prompts=prompts, coef=coef, max_tokens=max_tokens, qwen=True, orig=False)

# for l in all_langs:
#     if l == orig_lang or l == "italian":
#         continue

#     # l_controller = load_controller(llm, l, path=f'../all_gitignore/language_multi/')
#     l_controller = load_controller_translate(llm, l, orig_lang, path=path)

#     # out = test_concept_vector(l_controller, concept=f"coef: {coef}, lang: {l}", prompts=prompts, coef=coef, max_tokens=max_tokens, qwen=False, orig=False)
#     out = test_concept_vector(l_controller, concept=f"coef: {coef}, lang: {l}", prompts=prompts, coef=coef, max_tokens=max_tokens, qwen=True, orig=False)

Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31, -32, -33, -34, -35]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components         : 1

Using xRFM library for RFM-based steering vector extraction
Detector found

========================== + coef: 2.0, lang: korean Control (normal) ==========================
Assistant: In FIFA (the popular football/soccer video game series), there is **no official physical weight** for a football (soccer ball) because FIFA is a video game and does not use real-world physical objects.

However, if you're asking about the **weight of a standard football used in real-life football (soccer) games**, which is often referenced in FIFA games due to their realism, here is the standard:

✅ **Standard weight of a soccer ball (FIFA-style football):**  
- **400 

In [18]:
dataset = utils.multilingual_dataset(llm, source_lang, target_lang, n=600)

Loading FLORES-200 for english (eng_Latn) and korean (kor_Hang)...
train 1200


In [19]:
print(dataset)

{'korean': {'train': {'inputs': ("<|im_start|>user\nComplete the translation of the following statement in english to english. \nStatement: Stewart, Gordon, Kenseth, and Harvick round out the top-ten positions for the Drivers' Championship with four races remaining in the season.\nTranslation: Stewart, Gordon, Kenseth, and Harvick round out the top<|im_end|>\n<|im_start|>assistant", '<|im_start|>user\nComplete the translation of the following statement in english to english. \nStatement: The invention of spoke wheels made Assyrian chariots lighter, faster, and better prepared to outrun soldiers and other chariots.\nTranslation: The invention of spoke wheels made Assyrian chariots lighter, faster<|im_end|>\n<|im_start|>assistant', '<|im_start|>user\nComplete the translation of the following statement in english to korean. \nStatement: Tourists may visit different landmarks of a particular country or they may simply opt to focus on just one area.\nTranslation: 관광객들은 특정 국가의 여러 명소를 방문할 수 있